# Music Annotation Project 59R5
## Expanded Multi-Scale Retained-Region Challenger

**Runtime:** GPU recommended. Select a T4 GPU and run all.

59R5 is a development-only challenger. It consumes the Notebook 59R4 evidence and optional 59R4A adjudications, excludes `REGION_FRAGMENT` from supervised fitting, keeps low-support classes in the audit, and compares:

1. scalar observable baseline;
2. local 33×33 CNN;
3. multi-scale 33×33 / 97×97 / 257×257 CNN;
4. multi-scale CNN with observable local region merging.

Primary semantic classes:

`ANNOTATION / PRINT / MIXED / PAGE_SCAN_EDGE / UNCERTAIN`

The annotation-family head remains diagnostic only.

No model from this notebook has promotion authority. Page identity, raw coordinates, source-group identity, sampling-family names, 59R3 flags and clean/truth-derived inputs are prohibited predictors.


In [13]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy scikit-learn opencv-python-headless torch torchvision matplotlib pyarrow joblib


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, random, shutil, zipfile, sys, platform, gc
import numpy as np
import pandas as pd
import cv2, torch, joblib
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
R59R4_ROOT=ROOT/'notebook59R4_multiscale_region_semantics'
R59R4A_ROOT=ROOT/'region_semantic_adjudication_59R4A'
OUTPUT_ROOT=ROOT/'region_semantic_challenger_59R5'
SOURCE_59R4_RUN_OVERRIDE=None
SOURCE_59R4A_RUN_OVERRIDE=None
ADJUDICATION_CSV_OVERRIDE=None
SEED=42; EPOCHS=15; BATCH=32; LR=1e-3; WEIGHT_DECAY=1e-4; FOLDS=5
MIN_CLASS_REGIONS=10; MIN_CLASS_PAGES=5
MERGE_GAP=18; MERGE_DILATION=9
SCALES=[33,97,257]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_59R5_expanded_multiscale'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MODELS=OUT/'models'; VISUAL=OUT/'visuals'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MODELS,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(ok,msg):
    if not ok:raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def reset_seed(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def find_unique_below(root,name,label,required=True):
    hits=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()})
    if not hits and not required:return None
    require(len(hits)==1,f'{label}: expected one {name}, found {len(hits)}: {hits[:20]}')
    return hits[0]

R4=Path(SOURCE_59R4_RUN_OVERRIDE) if SOURCE_59R4_RUN_OVERRIDE else latest_complete(R59R4_ROOT,['tables/out_of_fold_predictions.csv','tables/model_summary.csv','audit/notebook59R4_verdict.json'])
require(R4 is not None,'No complete 59R4 run found.')
R4A=Path(SOURCE_59R4A_RUN_OVERRIDE) if SOURCE_59R4A_RUN_OVERRIDE else latest_complete(R59R4A_ROOT,['tables/review_adjudications.csv','notebook59R4A_verdict.json'])
PRED_PATH=R4/'tables'/'out_of_fold_predictions.csv'
DECISIONS_PATH=find_unique_below(R4,'notebook66_completed_snapshot.csv','Notebook 66 decisions')
REGIONS_PATH=find_unique_below(R4,'retained_regions_clustered.csv','59R3 retained regions',required=False)
if REGIONS_PATH is None: REGIONS_PATH=find_unique_below(R4,'reviewed_regions_joined.csv','59R4 region dataset')
ADJ_PATH=Path(ADJUDICATION_CSV_OVERRIDE) if ADJUDICATION_CSV_OVERRIDE else ((R4A/'tables'/'review_adjudications.csv') if R4A is not None else None)
if ADJ_PATH is not None and not ADJ_PATH.exists():ADJ_PATH=None
print('59R4:',R4);print('59R4A:',R4A);print('Decisions:',DECISIONS_PATH);print('Regions:',REGIONS_PATH);print('Adjudications:',ADJ_PATH);print('Device:',DEVICE);print('Output:',OUT)


59R4: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4
59R4A: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_093214_59R4A_targeted_adjudication
Decisions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/notebook66_completed_snapshot.csv
Regions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/reviewed_regions_joined.csv
Adjudications: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_093214_59R4A_targeted_adjudication/tables/review_adjudications.csv
Device: cuda
Output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_challenger_59R5/20261115_100624_59R5_expanded_multiscale


In [ ]:
pred=pd.read_csv(PRED_PATH,dtype=str).fillna('')
dec=pd.read_csv(DECISIONS_PATH,dtype=str).fillna('')
regions=pd.read_csv(REGIONS_PATH,low_memory=False)
for f,n in [(dec,'review decisions'),(pred,'59R4 predictions'),(regions,'region table')]:require('region_id' in f.columns,f'{n} lacks region_id')
completed=dec.merge(regions,on='region_id',how='left',validate='one_to_one',suffixes=('','_region')).merge(pred,on='region_id',how='left',validate='one_to_one',suffixes=('','_59R4'))
completed['original_semantic_label']=completed.semantic_label.astype(str)
completed['original_annotation_family']=completed.annotation_family.astype(str)
if ADJ_PATH is not None:
    adj=pd.read_csv(ADJ_PATH,dtype=str).fillna('')
    require({'region_id','adjudicated_semantic_label'}.issubset(adj.columns),'Adjudication schema invalid')
    require(not adj.region_id.duplicated().any(),'Duplicate adjudication region_id')
    completed=completed.merge(adj,on='region_id',how='left',validate='one_to_one')
    use=completed.adjudicated_semantic_label.astype(str).str.strip().ne('')
    completed.loc[use,'semantic_label']=completed.loc[use,'adjudicated_semantic_label']
    if 'adjudicated_annotation_family' in completed:
        uf=completed.adjudicated_annotation_family.astype(str).str.strip().ne('')
        completed.loc[uf,'annotation_family']=completed.loc[uf,'adjudicated_annotation_family']
completed['semantic_label']=completed.semantic_label.astype(str).str.upper().str.strip()
completed['annotation_family']=completed.annotation_family.astype(str).str.upper().str.strip()
completed['fit_exclusion_reason']=''
completed.loc[completed.semantic_label.eq('REGION_FRAGMENT'),'fit_exclusion_reason']='REGION_FRAGMENT'
completed.loc[completed.semantic_label.eq('MASK_ERROR'),'fit_exclusion_reason']='MASK_ERROR'

# Support audit and minimum-support gate.
support=completed.groupby('semantic_label',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
support['eligible_for_supervised_fit']=(support.regions>=MIN_CLASS_REGIONS)&(support.pages>=MIN_CLASS_PAGES)&~support.semantic_label.isin(['REGION_FRAGMENT','MASK_ERROR'])
support.to_csv(TABLES/'semantic_class_support.csv',index=False)
eligible_classes=support.loc[support.eligible_for_supervised_fit,'semantic_label'].tolist()
completed.loc[~completed.semantic_label.isin(eligible_classes)&completed.fit_exclusion_reason.eq(''),'fit_exclusion_reason']='INSUFFICIENT_CLASS_SUPPORT'
fit=completed[completed.fit_exclusion_reason.eq('')].copy()
require(fit.semantic_label.nunique()>=3,f'Fewer than three supported semantic classes: {eligible_classes}')
require(fit.page.nunique()>=10,'Too few pages for grouped OOF')
completed.to_csv(INPUTS/'effective_review_evidence.csv',index=False)
print(support.to_string(index=False));print('Fit rows:',len(fit),'pages:',fit.page.nunique(),'classes:',eligible_classes)


In [ ]:
# Predictor firewall and path resolution.
PROHIBITED_EXACT={'page','region_id','source_group','sampling_family','semantic_label','annotation_family','original_semantic_label','original_annotation_family','review_order','review_id','reviewed_utc','cluster_id','x','y','x0','y0','x1','y1','x_fraction','y_fraction'}
PROHIBITED_PATTERNS=['flag_','_flag_','human','truth','clean','outcome','failure','pred_','prob_','sampling_family','source_group','page_','adjudicat','review']

def first_col(frame,names):return next((c for c in names if c in frame.columns),None)
def numeric(names,default=np.nan):
    c=first_col(fit,names);return pd.to_numeric(fit[c],errors='coerce') if c else pd.Series(default,index=fit.index,dtype=float)
fit['_x0']=numeric(['x0','bbox_x0','x','left']);fit['_y0']=numeric(['y0','bbox_y0','y','top']);fit['_x1']=numeric(['x1','bbox_x1','right']);fit['_y1']=numeric(['y1','bbox_y1','bottom'])
if fit['_x1'].isna().all():fit['_x1']=fit['_x0']+numeric(['bbox_w','w','width'])
if fit['_y1'].isna().all():fit['_y1']=fit['_y0']+numeric(['bbox_h','h','height'])
require(fit[['_x0','_y0','_x1','_y1']].notna().all().all(),'Region geometry is incomplete')

# ============================================================
# Resolve authoritative observed pages
#
# 59R5 must not assume that observed pages were physically
# copied into the 59R4 run.
#
# Resolution priority:
#
#   1. explicit durable image path already present in row;
#   2. Notebook 07 review-audit aligned_output;
#   3. ROOT/aligned exact page stem;
#   4. ROOT/source/annotated_only exact page stem;
#
# Do not search 59R4 visual/contact-sheet outputs as if they
# were authoritative page inputs.
# ============================================================

SUPPORTED_IMAGES = {
    '.png',
    '.jpg',
    '.jpeg',
    '.tif',
    '.tiff',
    '.bmp',
}


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def image_map(
    folder,
):

    folder = Path(
        folder
    )

    if not folder.exists():

        return {}


    result = {}


    for path in folder.iterdir():

        if (
            path.is_file()
            and path.suffix.lower()
            in SUPPORTED_IMAGES
        ):

            stem = path.stem


            require(
                stem not in result,
                (
                    f'Duplicate page stem {stem!r} '
                    f'in {folder}.'
                ),
            )


            result[
                stem
            ] = path


    return result


def existing_image_path(
    value,
):

    if value is None:

        return None


    text = str(
        value
    ).strip()


    if (
        not text
        or text.lower()
        in {
            'nan',
            'none',
        }
    ):

        return None


    path = Path(
        text
    )


    if (
        path.is_file()
        and path.suffix.lower()
        in SUPPORTED_IMAGES
    ):

        return path


    # Relative to project root.

    candidate = (
        ROOT
        / path
    )


    if (
        candidate.is_file()
        and candidate.suffix.lower()
        in SUPPORTED_IMAGES
    ):

        return candidate


    return None


# ------------------------------------------------------------
# 2. Direct authoritative page maps
# ------------------------------------------------------------

ALIGNED_MAP = image_map(
    ROOT
    / 'aligned'
)


ANNOTATED_ONLY_MAP = image_map(
    ROOT
    / 'source'
    / 'annotated_only'
)


print(
    'ROOT/aligned image pages:',
    len(
        ALIGNED_MAP
    )
)

print(
    'ROOT/source/annotated_only image pages:',
    len(
        ANNOTATED_ONLY_MAP
    )
)


# ------------------------------------------------------------
# 3. Notebook 07 review audits
#
# Notebook 07 reviewed pages record the aligned page used by
# the corrected-mask workflow as aligned_output.
# ------------------------------------------------------------

REVIEW_AUDIT_DIR = (
    ROOT
    / 'human_in_the_middle'
    / 'audit'
)


review_audits = []


if REVIEW_AUDIT_DIR.exists():

    for audit_path in sorted(
        REVIEW_AUDIT_DIR.glob(
            '*.json'
        )
    ):

        try:

            record = json.loads(
                audit_path.read_text()
            )

        except Exception as error:

            raise RuntimeError(
                'Unreadable Notebook 07 review audit: '
                f'{audit_path}: {error}'
            )


        if not isinstance(
            record,
            dict,
        ):

            raise RuntimeError(
                'Notebook 07 review audit root is not '
                f'an object: {audit_path}'
            )


        record[
            '_audit_path'
        ] = str(
            audit_path
        )


        review_audits.append(
            record
        )


print(
    'Notebook 07 review audit records:',
    len(
        review_audits
    )
)


# ------------------------------------------------------------
# 4. Build audit lookup by safe page identities
# ------------------------------------------------------------

AUDIT_BY_PAGE = {}


def add_page_identity(
    raw_identity,
    record,
):

    if raw_identity is None:

        return


    text = str(
        raw_identity
    ).strip()


    if not text:

        return


    try:

        page = Path(
            text
        ).stem

    except Exception:

        page = text


    if not page:

        return


    AUDIT_BY_PAGE.setdefault(
        page,
        []
    )


    if record not in AUDIT_BY_PAGE[
        page
    ]:

        AUDIT_BY_PAGE[
            page
        ].append(
            record
        )


for record in review_audits:

    # Direct page identifiers.

    for key in [
        'page',
        'page_id',
        'source_stem',
        'stem',
        'name',
    ]:

        if record.get(
            key
        ):

            add_page_identity(
                record[
                    key
                ],
                record,
            )


    # File paths can recover the page stem as well.

    for key in [
        'source',
        'source_file',
        'aligned_output',
        'corrected_mask_output',
    ]:

        if record.get(
            key
        ):

            add_page_identity(
                record[
                    key
                ],
                record,
            )


# ------------------------------------------------------------
# 5. Resolve Notebook 07 aligned page
# ------------------------------------------------------------

def resolve_notebook07_observed(
    page,
):

    records = (
        AUDIT_BY_PAGE.get(
            str(
                page
            ),
            []
        )
    )


    paths = []


    for record in records:

        path = existing_image_path(
            record.get(
                'aligned_output'
            )
        )


        if path is not None:

            paths.append(
                path.resolve()
            )


    paths = sorted(
        set(
            paths
        )
    )


    # If multiple historical audit records resolve for the
    # same page, require byte identity rather than picking one
    # by modification time.

    if len(
        paths
    ) > 1:

        hashes = {
            sha256_file(
                path
            )
            for path
            in paths
        }


        require(
            len(
                hashes
            )
            == 1,
            (
                f'{page}: multiple non-identical Notebook 07 '
                f'aligned_output pages resolve: {paths}'
            ),
        )


    return (
        paths[
            0
        ]
        if paths
        else None
    )


# ------------------------------------------------------------
# 6. Explicit image-path columns potentially carried through
# 59R3 / Notebook 66 / 59R4.
# ------------------------------------------------------------

ROW_IMAGE_COLUMNS = [
    'observed_path',
    'observed_image_path',
    'page_path',
    'page_image_path',
    'aligned_path',
    'aligned_output',
    'source_path',
    'source_file',
    'image_path',
]


# ------------------------------------------------------------
# 7. Main authoritative resolver
# ------------------------------------------------------------

def resolve_page(
    row,
):

    page = str(
        row[
            'page'
        ]
    ).strip()


    # A. Exact path already persisted in the joined evidence.

    for column in ROW_IMAGE_COLUMNS:

        if column in row.index:

            path = existing_image_path(
                row[
                    column
                ]
            )


            if path is not None:

                return (
                    path,
                    f'row:{column}',
                )


    # B. Notebook 07 reviewed aligned output.
    #
    # Prefer this over generic aligned copies because this
    # records the actual page associated with human review.

    path = resolve_notebook07_observed(
        page
    )


    if path is not None:

        return (
            path,
            'Notebook07:aligned_output',
        )


    # C. Project-level aligned image.

    if page in ALIGNED_MAP:

        return (
            ALIGNED_MAP[
P[
                page
            ],
            'ROOT/aligned',
        )


    # D. Original annotated-only input.
    #
    # This is the final legitimate fallback. Do not substitute
    # contact sheets, threshold ladders or 59R4 crops for the
    # observed page.

    if page in ANNOTATED_ONLY_MAP:

        return (
            ANNOTATED_ONLY_MAP[
                page
            ],
            'ROOT/source/annotated_only',
        )


    return (
        None,
        None,
    )


# ------------------------------------------------------------
# 8. Resolve each unique page only once
# ------------------------------------------------------------

page_resolution_rows = []

PAGE_PATHS = {}


for page in sorted(
    fit[
        'page'
    ]
    .astype(str)
    .unique()
):

    representative = (
        fit[
            fit[
                'page'
            ].astype(str)
            == page
        ]
        .iloc[
            0
        ]
    )


    path, resolution_source = resolve_page(
        representative
    )


    PAGE_PATHS[
        page
    ] = path


    page_resolution_rows.append({
        'page':
            page,

        'observed_path':
            (
                str(
                    path
                )
                if path is not None
                else ''
            ),

        'resolution_source':
            (
                resolution_source
                or ''
            ),

        'resolved':
            bool(
                path is not None
            ),
    })


page_resolution = pd.DataFrame(
    page_resolution_rows
)


page_resolution.to_csv(
    AUDIT
    / 'observed_page_resolution.csv',
    index=False,
)


missing_pages = (  page_resolution[
        ~page_resolution[
            'resolved'
        ]
    ]
    .copy()
)


if len(
    missing_pages
):

    print(
        '\nUnresolved observed pages:'
    )

    print(
        missing_pages[
            'page'
        ].to_list()
    )


require(
    len(
        missing_pages
    )
    == 0,
    (
        f'{len(missing_pages)} unique observed pages '
        'could not be resolved. '
        'See audit/observed_page_resolution.csv.'
    ),
)


# ------------------------------------------------------------
# 9. Attach resolved page path to every fitted region
# ------------------------------------------------------------

fit[
    '_observed_path'
] = (
    fit[
        'page'
    ]
    .astype(str)
    .map(
        PAGE_PATHS
    )
)


require(
    fit[
        '_observed_path'
    ].notna().all(),
    (
        'One or more fitted rows lack an observed page '
        'after page-level resolution.'
    ),
)


# ------------------------------------------------------------
# 10. Verify readability and geometry against each page
# ------------------------------------------------------------

page_identity_rows = []


for page, group in fit.groupby(
    'page',
    sort=True,
):

    page = str(
        page
    )


    path = Path(
        PAGE_PATHS[
            page
        ]
    )


    observed = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    require(
        observed is not None,
        (
            f'{page}: resolved observed page is unreadable: '
            f'{path}'
        ),
    )


    height, width = (
        observed.shape
    )


    x0 = group[
        '_x0'
    ].to_numpy(
        dtype=float
    )

    y0 = group[
        '_y0'
    ].to_numpy(
        dtype=float
    )

    x1 = group[
        '_x1'
    ].to_numpy(
        dtype=float
    )

    y1 = group[
        '_y1'
    ].to_numpy(
        dtype=float
    )


    require(
        np.isfinite(
            np.column_stack(
                [
                    x0,
                    y0,
                    x1,
                    y1,
                ]
            )
        ).all(),
        (
            f'{page}: non-finite region geometry.'
        ),
    )


    invalid = (
        (
            x0 < 0
        )
        | (
            y0 < 0
        )
        | (
            x1 > width
        )
        | (
            y1 > height
        )
        | (
            x1 <= x0
        )
        | (
            y1 <= y0
        )
    )


    require(
        not invalid.any(),
        (
            f'{page}: '
            f'{int(invalid.sum())} candidate regions '
            'do not fit inside the resolved observed page '
            f'of shape {observed.shape}.'
        ),
    )


    page_identity_rows.append({
        'page':
            page,

        'observed_path':
            str(
                path
            ),

        'height':
            int(
                height
            ),

        'width':
            int(
                width
            ),

        'region_rows':
            int(
                len(
                    group
                )
            ),

        'sha256':
            sha256_file(
                path
            ),
    })


pd.DataFrame(
    page_identity_rows
).to_csv(
    AUDIT
    / 'resolved_observed_page_identity.csv',
    index=False,
)


print(
    '\nObserved-page resolution sources:'
)


print(
    page_resolution[
        'resolution_source'
    ].value_counts()
)


print(
    '\nResolved',
    len(
        PAGE_PATHS
    ),
    'unique observed pages for',
    len(
        fit
    ),
    'fitted regions.'
)

# Numeric scalar baseline excludes all identities, coordinates, automatic flags, labels and post-outcome fields.
scalar=[]
for c in fit.select_dtypes(include=[np.number,bool]).columns:
    lc=c.lower()
    if c in PROHIBITED_EXACT or c.startswith('_'):continue
    if any(p in lc for p in PROHIBITED_PATTERNS):continue
    scalar.append(c)
require(len(scalar)>0,'No scalar predictors remain after firewall')
firewall={'eligible_scalar_predictors':scalar,'prohibited_exact':sorted(PROHIBITED_EXACT),'prohibited_patterns':PROHIBITED_PATTERNS,'raw_coordinates_excluded':True,'page_identity_excluded':True,'source_group_excluded':True,'59R3_flags_excluded':True,'truth_derived_inputs_excluded':True}
atomic_json(AUDIT/'predictor_firewall.json',firewall)
print('Scalar predictors:',scalar)


In [ ]:
# Observable image channels and local region merging.
PAGE_CACHE={}
def load_page(
    path,
):

    path = Path(
        path
    )


    require(
        path.is_file(),
        (
            'Observed page does not exist: '
            f'{path}'
        ),
    )


    if path not in PAGE_CACHE:

        image = cv2.imread(
            str(
                path
            ),
            cv2.IMREAD_GRAYSCALE,
        )


        require(
            image is not None,
            (
                'Observed page is unreadable: '
                f'{path}'
            ),
        )


        PAGE_CACHE[
            path
        ] = image


    return PAGE_CACHE[
        path
    ]

def region_mask(row,shape,merged=False):
    m=np.zeros(shape,np.uint8)
    x0,y0,x1,y1=map(int,[row._x0,row._y0,row._x1,row._y1]);x0=max(0,x0);y0=max(0,y0);x1=min(shape[1],x1);y1=min(shape[0],y1)
    m[y0:y1,x0:x1]=1
    if merged:
        # Merge neighbouring reviewed candidate boxes on the same page using only observable gap/proximity.
        same=fit[fit.page.astype(str)==str(row.page)]
        cx=(row._x0+row._x1)/2;cy=(row._y0+row._y1)/2
        for _,q in same.iterrows():
            horiz=max(0,max(row._x0,q._x0)-min(row._x1,q._x1));vert=max(0,max(row._y0,q._y0)-min(row._y1,q._y1))
            if horiz<=MERGE_GAP and vert<=MERGE_GAP:
                xa,ya,xb,yb=map(int,[q._x0,q._y0,q._x1,q._y1]);m[max(0,ya):min(shape[0],yb),max(0,xa):min(shape[1],xb)]=1
        m=cv2.dilate(m,np.ones((MERGE_DILATION,MERGE_DILATION),np.uint8))
    return m>0

def centre(row):return int(round((row._y0+row._y1)/2)),int(round((row._x0+row._x1)/2))
def crop_fixed(a,cy,cx,size,pad_mode='reflect'):
    half=size//2;pad=half+2
    if pad_mode=='constant':p=np.pad(a,pad,mode='constant')
    else:p=np.pad(a,pad,mode=pad_mode)
    yy=cy+pad;xx=cx+pad;return p[yy-half:yy+half+1,xx-half:xx+half+1]
def channels(row,size,merged=False):
    im=load_page(row._observed_path);cy,cx=centre(row);mask=region_mask(row,im.shape,merged)
    blur=cv2.GaussianBlur(im,(0,0),45);norm=cv2.divide(im,np.maximum(blur,1),scale=255)
    edge=cv2.Canny(im,70,170).astype(np.float32)/255
    dist=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5);dist=dist/max(float(dist.max()),1.0)
    valid=np.ones(im.shape,np.float32)
    arrays=[im.astype(np.float32)/255,norm.astype(np.float32)/255,mask.astype(np.float32),dist.astype(np.float32),edge,valid]
    return np.stack([crop_fixed(a,cy,cx,size) for a in arrays]).astype(np.float32)

class RegionDataset(Dataset):
    def __init__(self,frame,indices,scales,merged=False):self.frame=frame.reset_index(drop=True);self.indices=np.asarray(indices);self.scales=scales;self.merged=merged
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i]);r=self.frame.iloc[j];xs=[torch.from_numpy(channels(r,s,self.merged)) for s in self.scales];return xs,torch.tensor(int(r._class_id),dtype=torch.long),j

def collate(batch):
    ns=len(batch[0][0]);xs=[torch.stack([b[0][k] for b in batch]) for k in range(ns)];y=torch.stack([b[1] for b in batch]);idx=torch.tensor([b[2] for b in batch]);return xs,y,idx

def loader(frame,idx,scales,merged,shuffle,seed):
    g=torch.Generator();g.manual_seed(seed);return DataLoader(RegionDataset(frame,idx,scales,merged),batch_size=BATCH,shuffle=shuffle,num_workers=0,collate_fn=collate,generator=g)
print('Tensor builder ready.')


In [ ]:
classes=sorted(fit.semantic_label.unique());class_to_id={c:i for i,c in enumerate(classes)};fit['_class_id']=fit.semantic_label.map(class_to_id).astype(int)
groups=fit.page.astype(str).to_numpy();splitter=GroupKFold(n_splits=min(FOLDS,len(np.unique(groups))))
splits=list(splitter.split(fit,fit._class_id,groups))
fold_map=[]
for f,(tr,te) in enumerate(splits,1):
    require(not(set(groups[tr])&set(groups[te])),f'Page leakage fold {f}')
    fold_map.extend({'region_id':fit.iloc[i].region_id,'page':fit.iloc[i].page,'fold':f} for i in te)
pd.DataFrame(fold_map).to_csv(INPUTS/'fold_map.csv',index=False)

class Branch(nn.Module):
    def __init__(self,nc=6):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
    def forward(self,x):return self.net(x).flatten(1)
class MultiCNN(nn.Module):
    def __init__(self,nbranches,nclasses):
        super().__init__();self.branches=nn.ModuleList([Branch() for _ in range(nbranches)]);self.head=nn.Sequential(nn.Linear(64*nbranches,96),nn.ReLU(),nn.Dropout(.15),nn.Linear(96,nclasses))
    def forward(self,xs):return self.head(torch.cat([b(x) for b,x in zip(self.branches,xs)],1))

def fit_predict_cnn(scales,merged,arm):
    oof=np.full(len(fit),-1,int);probs=np.full((len(fit),len(classes)),np.nan,float);hist=[]
    for f,(tr,te) in enumerate(splits,1):
        reset_seed();model=MultiCNN(len(scales),len(classes)).to(DEVICE)
        counts=np.bincount(fit.iloc[tr]._class_id,minlength=len(classes));weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE)
        loss_fn=nn.CrossEntropyLoss(weight=weights);opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
        for ep in range(EPOCHS):
            model.train();total=0
            for xs,y,_ in loader(fit,tr,scales,merged,True,SEED+f):
                xs=[x.to(DEVICE) for x in xs];y=y.to(DEVICE);opt.zero_grad(set_to_none=True);loss=loss_fn(model(xs),y);loss.backward();opt.step();total+=loss.item()*len(y)
            hist.append({'arm':arm,'fold':f,'epoch':ep+1,'loss':total/len(tr)})
        model.eval()
        with torch.no_grad():
            for xs,y,idx in loader(fit,te,scales,merged,False,SEED):
                p=torch.softmax(model([x.to(DEVICE) for x in xs]),1).cpu().numpy();probs[idx.numpy()]=p;oof[idx.numpy()]=p.argmax(1)
        path=MODELS/f'{arm}_fold{f}.pt';torch.save(model.state_dict(),path);del model;gc.collect();
        if DEVICE=='cuda':torch.cuda.empty_cache()
    require((oof>=0).all() and np.isfinite(probs).all(),f'Incomplete OOF for {arm}')
    return oof,probs,pd.DataFrame(hist)

# Scalar baseline.
X=fit[scalar].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
scalar_oof=np.full(len(fit),-1,int);scalar_probs=np.full((len(fit),len(classes)),np.nan)
for f,(tr,te) in enumerate(splits,1):
    model=Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler()),('clf',LogisticRegression(max_iter=5000,class_weight='balanced',random_state=SEED))])
    model.fit(X.iloc[tr],fit.iloc[tr]._class_id);scalar_oof[te]=model.predict(X.iloc[te]);scalar_probs[te]=model.predict_proba(X.iloc[te]);joblib.dump(model,MODELS/f'scalar_fold{f}.joblib')
local_oof,local_probs,h1=fit_predict_cnn([33],False,'local33')
multi_oof,multi_probs,h2=fit_predict_cnn(SCALES,False,'multiscale33_97_257')
merged_oof,merged_probs,h3=fit_predict_cnn(SCALES,True,'multiscale_merged')
pd.concat([h1,h2,h3],ignore_index=True).to_csv(TABLES/'training_history.csv',index=False)


In [ ]:
y=fit._class_id.to_numpy();arms={'scalar':(scalar_oof,scalar_probs),'local33':(local_oof,local_probs),'multiscale33_97_257':(multi_oof,multi_probs),'multiscale_merged':(merged_oof,merged_probs)}
summary=[];metrics=[];oof=fit[['region_id','page','semantic_label','annotation_family','original_semantic_label']].copy()
for arm,(pred,prob) in arms.items():
    summary.append({'arm':arm,'balanced_accuracy':balanced_accuracy_score(y,pred),'macro_f1':f1_score(y,pred,average='macro'),'weighted_f1':f1_score(y,pred,average='weighted'),'rows':len(y),'pages':fit.page.nunique()})
    report=classification_report(y,pred,labels=list(range(len(classes))),target_names=classes,output_dict=True,zero_division=0)
    for c in classes:metrics.append({'arm':arm,'class':c,**{k:report[c][k] for k in ['precision','recall','f1-score','support']}})
    oof[f'pred_{arm}']=[classes[i] for i in pred]
    oof[f'confidence_{arm}']=prob.max(1)
    for i,c in enumerate(classes):oof[f'p_{arm}_{c}']=prob[:,i]
    cm=confusion_matrix(y,pred,labels=range(len(classes)))
    plt.figure(figsize=(8,7));plt.imshow(cm,cmap='Blues');plt.xticks(range(len(classes)),classes,rotation=45,ha='right');plt.yticks(range(len(classes)),classes);plt.xlabel('Predicted');plt.ylabel('Human/adjudicated');plt.title(f'59R5 OOF confusion: {arm}')
    for i in range(len(classes)):
        for j in range(len(classes)):plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.tight_layout();plt.savefig(VISUAL/f'confusion_{arm}.png',dpi=170);plt.close()
summary=pd.DataFrame(summary).sort_values('balanced_accuracy',ascending=False);summary.to_csv(TABLES/'model_summary.csv',index=False);pd.DataFrame(metrics).to_csv(TABLES/'class_metrics.csv',index=False);oof.to_csv(TABLES/'out_of_fold_predictions.csv',index=False)
print(summary.to_string(index=False))


In [ ]:
# Calibrated abstention analysis. No operational threshold is authorised here.
best=summary.iloc[0].arm;pred,prob=arms[best]
rows=[]
for t in np.linspace(.50,.99,50):
    mx=prob.max(1);decision=np.where(mx>=t,pred,-1);selected=decision>=0
    rows.append({'arm':best,'confidence_threshold':float(t),'selected':int(selected.sum()),'coverage':float(selected.mean()),'balanced_accuracy_selected':balanced_accuracy_score(y[selected],decision[selected]) if selected.sum() and len(np.unique(y[selected]))>1 else np.nan,'macro_f1_selected':f1_score(y[selected],decision[selected],average='macro') if selected.sum() else np.nan,'errors':int(np.sum(decision[selected]!=y[selected]))})
abst=pd.DataFrame(rows);abst.to_csv(TABLES/'abstention_frontier.csv',index=False)
plt.figure(figsize=(8,6));plt.plot(abst.coverage,abst.balanced_accuracy_selected,marker='o');plt.xlabel('Coverage');plt.ylabel('Balanced accuracy among selected');plt.title(f'59R5 development abstention frontier: {best}');plt.tight_layout();plt.savefig(VISUAL/'abstention_frontier.png',dpi=170);plt.close()

# Family diagnostic head: fit only annotation/mixed rows and sufficiently supported families.
af=fit[fit.semantic_label.isin(['ANNOTATION','MIXED']) & ~fit.annotation_family.isin(['','NOT_APPLICABLE'])].copy()
af_support=af.groupby('annotation_family',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'));af_support['eligible']=(af_support.regions>=8)&(af_support.pages>=4);af_support.to_csv(TABLES/'annotation_family_support.csv',index=False)
# Keep family evaluation descriptive if support is too small; semantic ownership remains primary.
print('Best semantic arm:',best)
print('Annotation-family support:');print(af_support.to_string(index=False))


In [ ]:
# Diagnostic contact sheets for wins, regressions, uncertain/page-edge cases.
def contact(rows,name,n=20):
    rows=rows.head(n)
    if len(rows)==0:return
    cols=4;rr=int(np.ceil(len(rows)/cols));fig,axes=plt.subplots(rr,cols,figsize=(16,5*rr),squeeze=False)
    for ax in axes.ravel():ax.axis('off')
    for ax,(_,r) in zip(axes.ravel(),rows.iterrows()):
        im=load_page(r._observed_path);cy,cx=centre(r);size=257;crop=crop_fixed(im,cy,cx,size);ax.imshow(crop,cmap='gray',vmin=0,vmax=255);ax.set_title(f"{r.page} | human={r.semantic_label} | scalar={r.pred_scalar} | multi={r.pred_multi} | merged={r.pred_merged}",fontsize=8);ax.axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=150,bbox_inches='tight');plt.close()
view=fit.copy();view['pred_scalar']=[classes[i] for i in scalar_oof];view['pred_multi']=[classes[i] for i in multi_oof];view['pred_merged']=[classes[i] for i in merged_oof]
contact(view[(view.pred_scalar!=view.semantic_label)&(view.pred_multi==view.semantic_label)],'multiscale_wins')
contact(view[(view.pred_scalar==view.semantic_label)&(view.pred_multi!=view.semantic_label)],'multiscale_regressions')
contact(view[view.semantic_label.isin(['UNCERTAIN','PAGE_SCAN_EDGE'])],'uncertain_and_page_edges')
contact(view[(view.pred_multi!=view.semantic_label)&(view.pred_merged==view.semantic_label)],'region_merge_wins')
contact(view[(view.pred_multi==view.semantic_label)&(view.pred_merged!=view.semantic_label)],'region_merge_regressions')


In [ ]:
# Final development-only verdict and upload package.
bestrow=summary.iloc[0]
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'59R5','development_only':True,'promotion_authority':False,'source_59R4_run':str(R4),'source_59R4A_run':str(R4A) if R4A else None,'adjudication_used':ADJ_PATH is not None,'rows_total_evidence':len(completed),'rows_supervised_fit':len(fit),'pages':int(fit.page.nunique()),'classes_fitted':classes,'classes_audit_only':support.loc[~support.eligible_for_supervised_fit,'semantic_label'].tolist(),'best_arm':str(bestrow.arm),'best_balanced_accuracy':float(bestrow.balanced_accuracy),'best_macro_f1':float(bestrow.macro_f1),'region_fragment_excluded':int((completed.fit_exclusion_reason=='REGION_FRAGMENT').sum()),'minimum_class_regions':MIN_CLASS_REGIONS,'minimum_class_pages':MIN_CLASS_PAGES,'raw_coordinates_excluded':True,'page_identity_excluded':True,'source_group_excluded':True,'clean_truth_used_as_input':False}
atomic_json(OUT/'notebook59R5_verdict.json',verdict)
(OUT/'executive_summary.txt').write_text(chr(10).join(['NOTEBOOK 59R5 EXPANDED MULTI-SCALE REGION CHALLENGER','',f'Rows fitted: {len(fit)} across {fit.page.nunique()} pages',f'Classes fitted: {classes}',f'Best arm: {bestrow.arm}',f'Balanced accuracy: {bestrow.balanced_accuracy:.6f}',f'Macro F1: {bestrow.macro_f1:.6f}','','Development evidence only. No correction policy, threshold or production promotion is authorised.']))
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'device':DEVICE,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()};atomic_json(OUT/'software_environment.json',env)

artefacts=[OUT/'executive_summary.txt',OUT/'notebook59R5_verdict.json',TABLES/'model_summary.csv',TABLES/'class_metrics.csv',TABLES/'semantic_class_support.csv',TABLES/'out_of_fold_predictions.csv',TABLES/'abstention_frontier.csv',AUDIT/'predictor_firewall.json']
artefacts += sorted(VISUAL.glob('*.png'))
manifest=[]
for p in artefacts:
    require(p.exists(),f'Missing upload artefact {p}');dst=UPLOAD/p.name;shutil.copy2(p,dst);manifest.append({'file':p.name,'sha256':sha256_file(dst),'bytes':dst.stat().st_size})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=OUT/'notebook59R5_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():z.write(p,p.name)
print('59R5 complete:',OUT);print('Upload bundle:',zip_path)
